# ARC-AGI-3 STaR LoRA — Colab QLoRA training (A100 40GB)

Runs the same train_lora.py as the Kaggle kernel but in QLORA mode
(4-bit NF4 base) so the 27B fits in 40GB. Expected: ~3-4h for
SUBSAMPLE=1200, 1 epoch.

**Runtime: pick A100 (high-RAM) via Runtime > Change runtime type.**
L4-24GB likely OOMs at seq 8192 (vocab 248k logits); if stuck on L4,
set MAX_LEN to 4096 below (halves logit memory).

You need your Kaggle `credentials.json` (the OAuth one from
`~/.kaggle/credentials.json`) — the notebook asks for it in cell 2.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'peft==0.17.0', 'accelerate>=1.6', 'datasets>=3.0',
                       'transformers>=4.56.0', 'bitsandbytes>=0.45.0',
                       'kaggle>=1.7'])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'pick a GPU runtime (A100 recommended)'

In [ ]:
import json, os, pathlib
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
cred = kaggle_dir / 'credentials.json'
if not cred.exists():
    from google.colab import files
    print('upload your ~/.kaggle/credentials.json')
    up = files.upload()
    name = next(iter(up))
    cred.write_bytes(up[name])
print('kaggle creds:', cred)
os.chmod(cred, 0o600)
# sanity: the CLI should answer without interactive login
import subprocess
r = subprocess.run(['kaggle', 'datasets', 'list', '-m', '--max-size', '1'],
                   capture_output=True, text=True)
print((r.stdout or r.stderr)[:300])

In [ ]:
import os, pathlib, subprocess
BASE = pathlib.Path('/content/base_27b')
SFTD = pathlib.Path('/content/sft_data')
if not (BASE / 'config.json').exists():
    BASE.mkdir(exist_ok=True)
    print('downloading bf16 base (~44GB, ~20-60min)...')
    subprocess.check_call(['kaggle', 'datasets', 'download',
                           'rahim3/qwen3-8-27b-bf16', '--unzip',
                           '-p', str(BASE)])
if not (SFTD / 'packed_sft.jsonl').exists():
    SFTD.mkdir(exist_ok=True)
    subprocess.check_call(['kaggle', 'datasets', 'download',
                           'takumuhata/taaf-duck-sft-v1', '--unzip',
                           '-p', str(SFTD)])
print('base files:', len(list(BASE.glob('*.safetensors'))), 'shards')
print('sft files:', sorted(p.name for p in SFTD.iterdir()))

In [ ]:
import os, subprocess, sys
env = dict(os.environ)
env.update({
    'BASE_MODEL': '/content/base_27b',
    'SFT_JSONL': '/content/sft_data/packed_sft.jsonl',
    'OUT_DIR': '/content/lora_adapter',
    'EPOCHS': '1',
    'SUBSAMPLE': '1200',
    'LR': '1e-4',
    'MAX_LEN': '8192',
    'QLORA': '1',
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
})
train = '/content/sft_data/train_lora.py'
assert os.path.exists(train)
proc = subprocess.run([sys.executable, train], env=env, text=True)
assert proc.returncode == 0, 'train_lora failed'
import pathlib
print('adapter files:', sorted(p.name for p in
      pathlib.Path('/content/lora_adapter').iterdir()))

In [ ]:
import json, pathlib, subprocess
ad = pathlib.Path('/content/lora_adapter')
assert (ad / 'adapter_config.json').exists() and \
       (ad / 'adapter_model.safetensors').exists(), 'adapter incomplete'
(ad / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'taaf-duck-lora-v1',
    'id': 'takumuhata/taaf-duck-lora-v1',
    'licenses': [{'name': 'other'}],
}))
r = subprocess.run(['kaggle', 'datasets', 'status',
                    'takumuhata/taaf-duck-lora-v1'],
                   capture_output=True, text=True)
if r.returncode == 0:
    subprocess.check_call(['kaggle', 'datasets', 'version', '-p', str(ad),
                           '-m', 'qlora adapter (colab a100)'])
else:
    subprocess.check_call(['kaggle', 'datasets', 'create', '-p', str(ad)])
print('uploaded -> takumuhata/taaf-duck-lora-v1')
print('next: on the Devin side run arc-agi-3/sat_lora_pipeline.sh step 3'
      ' (or just push submit_ag3_lora) to eval the adapter')